#  PDF to Text

In [1]:
# Paths and snapshot
import os
from pathlib import Path

# Base data path (relative to notebook)
data_path = Path('01_programas')
snapshot = 'ss_20251020'

# PDF source folder and TXT output folder (under the snapshot directory)
pdf_path = data_path / snapshot / 'pdf'
txt_path = data_path / snapshot / 'txt'

# Ensure both paths exist (creates them if missing)
pdf_path.mkdir(parents=True, exist_ok=True)
txt_path.mkdir(parents=True, exist_ok=True)

print("pdf_path:", pdf_path.resolve())
print("txt_path:", txt_path.resolve())


pdf_path: G:\My Drive\Projects\2025\presidential\01_programas\ss_20251020\pdf
txt_path: G:\My Drive\Projects\2025\presidential\01_programas\ss_20251020\txt


In [2]:
# Functions for processing PDFs and main loop
from pathlib import Path

def get_pdf_files(pdf_path):
    """Return a sorted list of .pdf Path objects in pdf_path."""
    return sorted([p for p in Path(pdf_path).iterdir() if p.is_file() and p.suffix.lower() == '.pdf'])

def _import_pypdf():
    """Try to import pypdf and return the PdfReader class or None on failure."""
    try:
        from pypdf import PdfReader
        return PdfReader
    except Exception as e:
        print("pypdf not available:", e)
        return None

def extract_text_from_pdf(pdf_file, PdfReaderCls):
    """Extract text using PdfReaderCls. Return (ok: bool, content: str)."""
    try:
        reader = PdfReaderCls(str(pdf_file))
        texts = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
                texts.append(text)
        content = "\n\n".join(texts).strip()
        if not content:
            return False, f"[No extractable text found in {pdf_file.name}]"
        return True, content
    except Exception as e:
        return False, f"[Error extracting text from {pdf_file.name}: {e}]"

def write_output(out_file, content):
    out_file.write_text(content, encoding='utf-8')
    print("Wrote:", out_file)

def process_pdfs(pdf_path, txt_path, snapshot, overwrite=True):
    """Process all PDFs in pdf_path and write outputs to txt_path.

    Returns a list of Path objects written.
    """
    pdf_files = get_pdf_files(pdf_path)

    if not pdf_files:
        print("No PDF files found in", pdf_path)
        return []

    PdfReaderCls = _import_pypdf()
    written = []

    for pdf_file in pdf_files:
        base = pdf_file.stem
        out_name = f"{base}_{snapshot}.txt"
        out_file = Path(txt_path) / out_name

        if out_file.exists() and not overwrite:
            print("Skipping existing:", out_file)
            continue

        if PdfReaderCls is not None:
            ok, content = extract_text_from_pdf(pdf_file, PdfReaderCls)
            # if ok==False, content holds diagnostic message which we'll write
        else:
            content = f"[Placeholder file created for {pdf_file.name} — pypdf not installed]\n"

        write_output(out_file, content)
        written.append(out_file)

    return written

# Run processing once (change overwrite flag as needed)
_written = process_pdfs(pdf_path, txt_path, snapshot)
print(f"Processed {_written.__len__()} files")


Wrote: 01_programas\ss_20251020\txt\EDUARDO-ANTONIO-ARTES-BRICHETTI_ss_20251020.txt
Wrote: 01_programas\ss_20251020\txt\EVELYN-MATTHEI-FORNET_ss_20251020.txt
Wrote: 01_programas\ss_20251020\txt\FRANCO-PARISI-FERNANDEZ_ss_20251020.txt
Wrote: 01_programas\ss_20251020\txt\HAROLD-MAYNE-NICHOLLS-SECUL_ss_20251020.txt
Wrote: 01_programas\ss_20251020\txt\JEANNETTE-JARA-ROMAN_ss_20251020.txt
Wrote: 01_programas\ss_20251020\txt\JOHANNES-KAISER-BARENTS-VON-HOHENHAGEN_ss_20251020.txt
Wrote: 01_programas\ss_20251020\txt\JOSE-ANTONIO-KAST-RIST_ss_20251020.txt
Wrote: 01_programas\ss_20251020\txt\MARCO-ANTONIO-ENRIQUEZ-OMINAMI-GUMUCIO_ss_20251020.txt
Processed 8 files
